In [2]:
import pandas as pd
# import scanpy as sc
import numpy as np
import anndata as ad
import json

from choose_protein_coding import list_of_protein_coding_genes

In [3]:
# configure
raw_data_dir = "../data/raw_tsv_data"
filename = "TCGA-OV.star_tpm"

data_for_mlp_dir = "../data/0_data_for_mlp"
data_for_scgpt_dir = "../data/0_adata_for_scgpt"

gene_info_file = "../data/gene_info.csv"

# for MLP

In [4]:
df_original = pd.read_csv(f'{raw_data_dir}/{filename}.tsv', delimiter='\t', index_col=False)

In [5]:
df = df_original.rename(columns={'Ensembl_ID': 'Unnamed: 0-1'})
df.head()

,Unnamed: 0-1,TCGA-24-1104-01A,TCGA-25-1320-01A,TCGA-24-1850-01A,TCGA-61-2101-01A,TCGA-23-1111-01A,TCGA-25-2396-01A,TCGA-30-1714-01A,TCGA-04-1347-01A,TCGA-59-2351-01A,...,TCGA-25-1632-01A,TCGA-25-2404-01A,TCGA-25-1323-01A,TCGA-13-0886-01A,TCGA-24-2262-01A,TCGA-36-1568-01A,TCGA-20-1685-01A,TCGA-61-1738-01A,TCGA-25-1623-01A,TCGA-13-0890-01A
0,ENSG00000000003.15,4.115449,6.141792,5.181945,4.949922,5.876416,5.640725,5.416573,5.904751,4.753861,...,6.314428,5.324141,6.025580,6.385897,5.016354,5.400456,6.663299,6.045410,4.588739,6.005872
1,ENSG00000000005.6,0.040963,0.088820,0.821384,0.065572,0.042784,0.221197,0.144569,0.516923,0.044324,...,0.052555,0.899330,0.818932,1.480472,0.230080,0.089769,0.123269,0.872395,0.228111,1.013641
2,ENSG00000000419.13,6.678187,7.287420,6.878219,6.975145,7.678456,7.151651,7.047638,7.668338,6.606639,...,6.897072,7.050592,7.446109,7.094548,6.436022,6.788650,7.097267,6.856525,7.514638,6.407983
3,ENSG00000000457.14,2.808570,3.479205,2.422071,2.680504,2.479334,2.411616,2.522884,2.050223,2.619789,...,2.657274,2.577610,2.795185,2.001514,2.940298,2.789312,2.645494,2.538464,2.365077,1.804343
4,ENSG00000000460.17,1.464459,3.600615,2.828002,2.509214,2.491186,2.372534,1.989575,1.448425,2.584674,...,2.213285,2.424841,2.896737,1.494262,2.750542,2.133827,2.813094,2.064676,2.409717,2.287354


In [6]:
# choose 01A for samples with duplicates

col_df = pd.DataFrame({"col": df.columns})

# prefix bez ostatniej litery
col_df["prefix"] = col_df["col"].str[:-1]

# suffix = ostatnia litera
col_df["suffix"] = col_df["col"].str[-1]

col_df["is_A"] = (col_df["suffix"] == "A").astype(int)
col_df = col_df.sort_values(
    ["prefix", "is_A"],
    ascending=[True, False]
)
selected_cols = col_df.drop_duplicates("prefix")["col"].tolist()

if 'Unnamed: 0-1' in selected_cols:
    selected_cols.remove('Unnamed: 0-1')
selected_cols.insert(0, 'Unnamed: 0-1')


In [ ]:
df = df[selected_cols]
df.columns = df.columns.str.split("-").str[:-1].str.join("-")

df = df.T
df.columns = df.iloc[0]   # pierwszy wiersz → nazwy kolumn
df = df.iloc[1:]

In [8]:
df.head()

Unnamed: 0,ENSG00000000003.15,ENSG00000000005.6,ENSG00000000419.13,ENSG00000000457.14,ENSG00000000460.17,ENSG00000000938.13,ENSG00000000971.16,ENSG00000001036.14,ENSG00000001084.13,ENSG00000001167.14,...,ENSG00000288661.1,ENSG00000288662.1,ENSG00000288663.1,ENSG00000288665.1,ENSG00000288667.1,ENSG00000288669.1,ENSG00000288670.1,ENSG00000288671.1,ENSG00000288674.1,ENSG00000288675.1
TCGA-04-1331,6.188982,0.399445,6.964604,3.087055,2.830357,1.821261,4.551959,6.209091,2.395529,4.821098,...,0.0,1.361488,0.913799,0.0,0.825297,0.0,3.266787,0.0,0.203765,1.194907
TCGA-04-1332,5.451359,0.706553,6.35779,2.366196,2.358199,2.625551,4.963719,5.289868,2.338767,3.958128,...,0.0,1.844386,0.57628,0.0,0.0,0.0,2.322044,0.0,0.057138,1.623867
TCGA-04-1337,4.715048,0.056167,7.952876,2.072312,1.397474,2.746227,3.988021,4.914899,3.064297,3.240314,...,0.0,0.983094,1.03485,0.0,0.971369,0.0,3.349705,0.0,0.021764,1.474618
TCGA-04-1338,4.427935,0.070389,6.422475,2.108324,1.537545,2.356904,4.106382,4.401173,2.92275,4.786476,...,0.0,0.0,0.271545,0.0,0.0,0.0,3.438173,0.0,0.011066,0.909351
TCGA-04-1341,4.718805,0.210888,6.403525,1.707348,1.402777,1.460323,2.099666,5.925715,2.307224,2.834731,...,0.0,0.569491,0.519139,0.0,0.56179,0.0,2.22657,0.0,0.013069,0.816067


In [9]:
df.columns = df.columns.str.split(".").str[0]

features = pd.read_csv(gene_info_file)
features = features[["feature_id", "feature_name"]]

id_to_symbol = dict(
    zip(features["feature_id"], features["feature_name"])
)

df = df.rename(columns=id_to_symbol)

df = df.loc[:, df.columns.notnull()]
df = df.loc[:, ~df.columns.duplicated()]

df.shape

(428, 60616)

In [10]:
# keep only protein-coding genes

gene_list = df.columns
gene_list = gene_list[1:]
protein_coding_gene_list = list_of_protein_coding_genes(gene_list)

df_filtered = df[ df.columns.intersection(protein_coding_gene_list) ]


In [11]:
df_filtered = df_filtered[~df_filtered.index.duplicated(keep="first")]


In [12]:
# save to file
df_filtered.to_csv(f"{data_for_mlp_dir}/{filename}.csv")


# for scGPT

In [13]:
# create anndata df

X = df_filtered.values.astype(np.float32)

obs = pd.DataFrame(index=df_filtered.index)
obs["sample"] = df_filtered.index

var = pd.DataFrame(index=df_filtered.columns)
var["gene_name"] = df_filtered.columns

adata = ad.AnnData(
    X=X,
    obs=obs,
    var=var
)

adata

AnnData object with n_obs × n_vars = 422 × 20260
    obs: 'sample'
    var: 'gene_name'

In [14]:
adata.write(f"{data_for_scgpt_dir}/adata_{filename}.h5ad")
